In [2]:
import numpy as np
import matplotlib.pyplot as plt
import requests

import torch
import torch.nn as nn

from transformers import BertModel, BertTokenizer

In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [4]:
# IMPORT 2 datasets
# Alice in wonderland
text = requests.get('https://www.gutenberg.org/cache/epub/11/pg11.txt').text
aliceTokens = torch.tensor(tokenizer.encode(text), dtype=torch.long)

# edgar allen poe
text = requests.get('https://www.gutenberg.org/cache/epub/2148/pg2148.txt').text
edgarTokens = torch.tensor(tokenizer.encode(text), dtype=torch.long)


Token indices sequence length is longer than the specified maximum sequence length for this model (40906 > 512). Running this sequence through the model will result in indexing errors


In [5]:
aliceTokens.shape

torch.Size([40906])

In [6]:
class BertForBinaryClassification(nn.Module):
    def __init__(self, num_labels=2):
        super(BertForBinaryClassification, self).__init__()

        # Load the pretrained BERT model
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        #classification head that converts 678-d pooled output into 2 final outuputs
        self.classifier = nn.Linear(768,2)
        self.dropout = nn.Dropout(.1) #10%

        #init the weights and biases
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask = None, token_type_ids = None):
        # fwd pas through the downloaded(pretrained) BERT
        outputs = self.bert(
            input_ids = input_ids,
            attention_mask = attention_mask,
            token_type_ids = token_type_ids)

        # extract the pooled output and apply dropout
        pooled_output  = self.dropout(outputs.pooler_output)

        # final push through classification layer
        logits = self.classifier(pooled_output)
        return logits

In [7]:
# create an instance of model and test it:
model = BertForBinaryClassification().to(device)
model

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForBinaryClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, e

Preparing to fine tune the model

In [8]:
num_training = 150
batch_size = 32
seq_len = 256

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-7)
loss_fun = nn.CrossEntropyLoss()

In [16]:
# create a batch of data
# this celll shows one step of a training


#first 16 rows of a batch are from alice and next 16 from Edgar
ixA = torch.randint(len(aliceTokens)-seq_len, size = (batch_size//2,))
ixE = torch.randint(len(edgarTokens)-seq_len, size = (batch_size//2,))
X = torch.concatenate(
    (aliceTokens[ixA[:,None] + torch.arange(seq_len)],
     edgarTokens[ixE[:,None] + torch.arange(seq_len)]), axis=0).to(device)

# and labels 
labels = torch.concatenate((
    torch.zeros(batch_size//2,dtype=torch.long),
    torch.ones(batch_size//2, dtype=torch.long)),
                           axis=0).to(device)

print(f'Data batch shape: {X.shape}')
print(f'Labels batch shape: {labels.shape}')

#fwd pass, get model preds, and report loss+accuracy
logits = model(X)
predLabels = torch.argmax(logits, dim=1)
loss = loss_fun(logits, labels).item()

print('\n Predicted labels:\n', predLabels)
print('Acutal labels:\n', labels)

print(f'\nLoss: {loss:.4f}')
print(f'\nAccuracy: {(predLabels == labels).sum().item()/batch_size}')

Data batch shape: torch.Size([32, 256])
Labels batch shape: torch.Size([32])


KeyboardInterrupt: 

Traing the model

In [11]:
losses = np.zeros(num_training)
accuracy = np.zeros(num_training)

for sampli in range(num_training):
    ixA = torch.randint(len(aliceTokens)-seq_len, size = (batch_size//2,))
    ixE = torch.randint(len(edgarTokens)-seq_len, size = (batch_size//2,))
    X = torch.concatenate(
        (aliceTokens[ixA[:,None] + torch.arange(seq_len)],
         edgarTokens[ixE[:,None] + torch.arange(seq_len)]), axis=0).to(device)
    labels = torch.concatenate((
        torch.zeros(batch_size//2,dtype=torch.long),
        torch.ones(batch_size//2, dtype=torch.long)),
                           axis=0).to(device)
    optimizer.zero_grad()
    logits = model(X)
    predLabels = torch.argmax(logits, dim=1)
    loss = loss_fun(logits, predLabels)

    losses[sampli] = loss.item()
    accuracy[sampli] = (predLabels == labels).sum().item() / batch_size
    loss.backward()
    optimizer.step()

    if sampli%50 == 0:
        print(f'Sample {sampli:4}/{num_training}, losses: {losses[sampli]:.2f}, accuracy: {accuracy[sampli]:.2f}')

Sample    0/150, losses: 0.39, accuracy: 0.50
Sample   50/150, losses: 0.17, accuracy: 0.50
Sample  100/150, losses: 0.07, accuracy: 0.50


In [12]:
labels

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1], device='mps:0')

In [19]:
# mean smoothing
def meansmooth(x, k=3):
    y = x+0 # copy of data
    w = (k-1)//2 # no of elemnts to avg on either side

    # loop over samples
    for i in range(w, len(x)-w):
        y[i] = x[i-w:i+w].mean() # centered mean
    return y

# smooth out the losses and accuracy of training and then visualise


In [ ]:
# Save the model
torch.save(model.state_dict(), 'bert_classifier_AliceVsEdgar.pt')
